# Portfolio VaR Backtesting
99% VaR for 60/40 S&P 500 / DAX portfolio with DCC vs CCC vs Static

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import yfinance as yf
from arch import arch_model
from scipy.optimize import minimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Chart style
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Crimson  = '#DC3545'
GoldC    = '#DAA520'

mpl.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})

In [ ]:
# Download data
tickers = ['^GSPC', '^GDAXI', '^VIX']

data = yf.download(tickers, start='2004-01-01', end='2024-12-31',
                   auto_adjust=True, progress=False)['Close']
data.columns = ['DAX', 'S&P 500', 'VIX']  # yfinance alphabetizes

# Reorder
data = data[['S&P 500', 'DAX', 'VIX']]
data = data.dropna()

# Percentage returns
returns = data[['S&P 500', 'DAX']].pct_change().dropna() * 100
vix = data['VIX'].loc[returns.index]

# Portfolio weights
w = np.array([0.6, 0.4])  # 60% S&P 500, 40% DAX

# Portfolio returns
port_ret = returns.values @ w

print(f'Sample: {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Observations: {len(returns)}')
print(f'Portfolio mean: {port_ret.mean():.4f}%, std: {port_ret.std():.4f}%')

In [ ]:
# Fit GARCH(1,1) for both indices
labels = ['S&P 500', 'DAX']
cond_vol = {}
std_resid = {}

for name in labels:
    am = arch_model(returns[name].dropna(), vol='Garch', p=1, q=1,
                    mean='Constant', dist='t')
    res = am.fit(disp='off')
    cond_vol[name] = res.conditional_volatility
    std_resid[name] = res.std_resid
    print(f'{name}: alpha={res.params["alpha[1]"]:.4f}, '
          f'beta={res.params["beta[1]"]:.4f}')

# DCC estimation
z_df = pd.DataFrame({l: std_resid[l] for l in labels}).dropna()
z = z_df.values
T, k = z.shape

Q_bar = np.corrcoef(z.T)
rho_bar = Q_bar[0, 1]
print(f'\nCCC correlation: {rho_bar:.4f}')

def dcc_loglik(params, z, Q_bar):
    a, b = params
    if a < 0 or b < 0 or a + b >= 1:
        return 1e10
    T, k = z.shape
    Q_t = Q_bar.copy()
    ll = 0.0
    for t in range(T):
        Q_t = (1 - a - b) * Q_bar + a * np.outer(z[t], z[t]) + b * Q_t
        d = np.sqrt(np.diag(Q_t))
        R_t = Q_t / np.outer(d, d)
        try:
            sign, logdet = np.linalg.slogdet(R_t)
            if sign <= 0:
                return 1e10
            R_inv = np.linalg.inv(R_t)
            ll += -0.5 * (logdet + z[t] @ R_inv @ z[t] - z[t] @ z[t])
        except np.linalg.LinAlgError:
            return 1e10
    return -ll

res_dcc = minimize(dcc_loglik, x0=[0.02, 0.95], args=(z, Q_bar),
                   method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-8})
a_dcc, b_dcc = res_dcc.x
print(f'DCC parameters: a = {a_dcc:.6f}, b = {b_dcc:.6f}')

In [ ]:
# Compute portfolio volatility and VaR under DCC, CCC, and Static
vol_df = pd.DataFrame({l: cond_vol[l] for l in labels}).loc[z_df.index]
ret_aligned = returns.loc[z_df.index]
vix_aligned = vix.loc[z_df.index]

port_ret_aligned = ret_aligned.values @ w

# Static covariance
cov_static = np.cov(ret_aligned.values.T)
vol_static = np.sqrt(w @ cov_static @ w)

# VaR at 99% (z = 2.326)
z_99 = stats.norm.ppf(0.01)  # -2.326

var_dcc = np.zeros(T)
var_ccc = np.zeros(T)
var_static = np.full(T, z_99 * vol_static)  # constant

Q_t = Q_bar.copy()
for t in range(T):
    Q_t = (1 - a_dcc - b_dcc) * Q_bar + a_dcc * np.outer(z[t], z[t]) + b_dcc * Q_t
    d = np.sqrt(np.diag(Q_t))
    R_t = Q_t / np.outer(d, d)

    sigma_t = vol_df.iloc[t].values
    D_t = np.diag(sigma_t)

    # DCC covariance
    H_dcc = D_t @ R_t @ D_t
    port_vol_dcc = np.sqrt(w @ H_dcc @ w)
    var_dcc[t] = z_99 * port_vol_dcc

    # CCC covariance
    H_ccc = D_t @ Q_bar @ D_t
    port_vol_ccc = np.sqrt(w @ H_ccc @ w)
    var_ccc[t] = z_99 * port_vol_ccc

print(f'VaR 99% computed for {T} observations')
print(f'DCC VaR range: [{var_dcc.min():.2f}%, {var_dcc.max():.2f}%]')
print(f'Static VaR: {var_static[0]:.2f}%')

In [ ]:
# Chart 1: VaR backtest
dates = z_df.index

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

# Portfolio returns as bars
ax.bar(dates, port_ret_aligned, width=1, color='#7BA7CC', alpha=0.5,
       label='Portfolio returns', linewidth=0)

# VaR lines
ax.plot(dates, var_dcc, color=MainBlue, linewidth=1.2, label='VaR 99% (DCC)')
ax.plot(dates, var_ccc, color=IDAred, linewidth=1.2, label='VaR 99% (CCC)')
ax.plot(dates, var_static, color='gray', linewidth=1.0, linestyle='--',
        label='VaR 99% (Static)')

# Highlight violations (returns below DCC VaR)
violations = port_ret_aligned < var_dcc
ax.scatter(dates[violations], port_ret_aligned[violations],
           color=Crimson, s=15, zorder=5, label='DCC violations',
           edgecolors='none')

ax.set_ylabel('Return / VaR (%)')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10),
          ncol=5, frameon=False, fontsize=9)

plt.tight_layout()
fig.savefig('../../../charts/ch5b_var_backtest.pdf',
            bbox_inches='tight', transparent=True, dpi=150)
plt.show()
print('Saved: charts/ch5b_var_backtest.pdf')

In [ ]:
# Chart 2: VaR by VIX regime
# Define regimes
vix_vals = vix_aligned.values
regime_labels_list = ['VIX < 15', '15-30', 'VIX > 30']

regime = np.where(vix_vals < 15, 0,
         np.where(vix_vals <= 30, 1, 2))

# Mean VaR and violation rate by regime and method
methods = ['DCC', 'CCC', 'Static']
var_arrays = [var_dcc, var_ccc, var_static]

mean_var_data = np.zeros((3, 3))  # regimes x methods
viol_rate_data = np.zeros((3, 3))

for j, (method, var_arr) in enumerate(zip(methods, var_arrays)):
    for r in range(3):
        mask = regime == r
        if mask.sum() > 0:
            mean_var_data[r, j] = np.abs(np.mean(var_arr[mask]))
            viol_rate_data[r, j] = np.mean(port_ret_aligned[mask] < var_arr[mask]) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
fig.patch.set_alpha(0)
ax1.patch.set_alpha(0)
ax2.patch.set_alpha(0)

x = np.arange(3)
bar_width = 0.25
colors = [MainBlue, IDAred, 'gray']

# Left: Mean |VaR|
for j, (method, color) in enumerate(zip(methods, colors)):
    ax1.bar(x + j * bar_width, mean_var_data[:, j], bar_width,
            color=color, alpha=0.85, label=method)

ax1.set_xticks(x + bar_width)
ax1.set_xticklabels(regime_labels_list)
ax1.set_ylabel('Mean |VaR| (%)')
ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
           ncol=3, frameon=False)

# Right: Violation rate
for j, (method, color) in enumerate(zip(methods, colors)):
    ax2.bar(x + j * bar_width, viol_rate_data[:, j], bar_width,
            color=color, alpha=0.85, label=method)

ax2.axhline(y=1.0, color='black', linestyle=':', linewidth=0.8,
            label='Expected (1%)')
ax2.set_xticks(x + bar_width)
ax2.set_xticklabels(regime_labels_list)
ax2.set_ylabel('Violation rate (%)')
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
           ncol=4, frameon=False)

# Remove top/right spines
for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig('../../../charts/ch5b_var_regime.pdf',
            bbox_inches='tight', transparent=True, dpi=150)
plt.show()
print('Saved: charts/ch5b_var_regime.pdf')

In [ ]:
# Violation rates and Christoffersen test
print('VaR 99% Backtest Results')
print('=' * 60)

def christoffersen_test(violations):
    """
    Christoffersen (1998) conditional coverage test.
    Tests both unconditional coverage and independence.
    violations: boolean array (True = violation)
    Returns: LR_uc, LR_ind, LR_cc, p_value_cc
    """
    T = len(violations)
    v = violations.astype(int)
    n1 = v.sum()       # number of violations
    n0 = T - n1        # number of non-violations
    pi_hat = n1 / T    # observed violation rate
    alpha = 0.01       # expected rate

    # Unconditional coverage test (Kupiec)
    if pi_hat == 0 or pi_hat == 1:
        LR_uc = np.nan
    else:
        LR_uc = -2 * (n0 * np.log(1 - alpha) + n1 * np.log(alpha)
                       - n0 * np.log(1 - pi_hat) - n1 * np.log(pi_hat))

    # Independence test
    # Count transitions
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        if v[t-1] == 0 and v[t] == 0: n00 += 1
        elif v[t-1] == 0 and v[t] == 1: n01 += 1
        elif v[t-1] == 1 and v[t] == 0: n10 += 1
        else: n11 += 1

    pi01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0
    pi2  = (n01 + n11) / (T - 1)

    if pi01 == 0 or pi01 == 1 or pi11 == 0 or pi11 == 1 or pi2 == 0 or pi2 == 1:
        LR_ind = np.nan
    else:
        LR_ind = -2 * ((n00 + n10) * np.log(1 - pi2) + (n01 + n11) * np.log(pi2)
                       - n00 * np.log(1 - pi01) - n01 * np.log(pi01)
                       - n10 * np.log(1 - pi11) - n11 * np.log(pi11))

    LR_cc = LR_uc + LR_ind if not (np.isnan(LR_uc) or np.isnan(LR_ind)) else np.nan
    p_uc  = 1 - stats.chi2.cdf(LR_uc, 1) if not np.isnan(LR_uc) else np.nan
    p_ind = 1 - stats.chi2.cdf(LR_ind, 1) if not np.isnan(LR_ind) else np.nan
    p_cc  = 1 - stats.chi2.cdf(LR_cc, 2) if not np.isnan(LR_cc) else np.nan

    return LR_uc, p_uc, LR_ind, p_ind, LR_cc, p_cc

for method, var_arr in zip(methods, var_arrays):
    violations = port_ret_aligned < var_arr
    n_viol = violations.sum()
    rate = n_viol / T * 100
    LR_uc, p_uc, LR_ind, p_ind, LR_cc, p_cc = christoffersen_test(violations)

    print(f'\n{method}:')
    print(f'  Violations: {n_viol}/{T} ({rate:.2f}%, expected 1.00%)')
    print(f'  Kupiec UC test:     LR = {LR_uc:.3f}, p = {p_uc:.4f}'
          f' {"REJECT" if p_uc < 0.05 else "OK"}')
    print(f'  Independence test:  LR = {LR_ind:.3f}, p = {p_ind:.4f}'
          f' {"REJECT" if p_ind < 0.05 else "OK"}' if not np.isnan(LR_ind)
          else f'  Independence test:  N/A')
    print(f'  Cond. Coverage:     LR = {LR_cc:.3f}, p = {p_cc:.4f}'
          f' {"REJECT" if p_cc < 0.05 else "OK"}' if not np.isnan(LR_cc)
          else f'  Cond. Coverage:     N/A')

## Results
- DCC-GARCH VaR adapts to changing market conditions, producing tighter VaR in calm periods and wider VaR during crises
- Static VaR overestimates risk in calm markets and underestimates it during stress
- Violation rates closer to 1% indicate better calibration; DCC typically achieves the best unconditional coverage
- The Christoffersen test reveals whether violations cluster (independence), which static VaR often fails
- VIX regime analysis shows that DCC VaR scales appropriately with implied volatility, while static VaR remains constant